In [1]:
!pip install pandas

In [9]:
!pip install -q "datasets<4.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 10.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [2]:
import pandas as pd

In [10]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset("rexarski/eli5_category", trust_remote_code=True)

# Combine all splits (train, validation1, validation2, test) into one DataFrame
df = pd.concat([ds[split].to_pandas() for split in ds], ignore_index=True)

# Keep answers that don't start with ** and are longer than 3 words
def clean_answers(answers):
    return [
        a.strip() for a in answers['text']
        if not a.strip().startswith('**') and len(a.split()) > 3
    ]

df['explanations'] = df['answers'].apply(clean_answers)

# Drop questions with no answers left
df = df[df['explanations'].str.len() > 0].reset_index(drop=True)

# Spread each question's explanations into their own columns
expl_cols = pd.DataFrame(df['explanations'].tolist())
expl_cols.columns = [f'explanation_{i+1}' for i in range(expl_cols.shape[1])]

comments_df = expl_cols
comments_df.head()

README.md:   0%|          | 0.00/12.6k [00:00<?, ?B/s]

eli5_category.py:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/91772 [00:00<?, ? examples/s]

Generating validation1 split:   0%|          | 0/5446 [00:00<?, ? examples/s]

Generating validation2 split:   0%|          | 0/2375 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5411 [00:00<?, ? examples/s]

,explanation_1,explanation_2,explanation_3,explanation_4,explanation_5,explanation_6,explanation_7,explanation_8,explanation_9,explanation_10,...,explanation_479,explanation_480,explanation_481,explanation_482,explanation_483,explanation_484,explanation_485,explanation_486,explanation_487,explanation_488
0,the rotation of the earth is not a constant. i...,The Earth's rotation is not regular. It varies...,Because the Earth's rotation is slowing. If yo...,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,Imagine you are out walking in the woods near ...,"By force. Historically, nations have defended ...",With a flag and an force/money to back the cla...,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,It's more that we're really good at picking up...,Probably because we are so adapted to human sk...,When was the last time you compared an Orc IRL...,It's a phenomenon known as the uncanny valley....,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,Anxiety/stress are the result of your sympathe...,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,In a 1080p screen each pixel is represented by...,Besides what people are saying about upscaling...,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [12]:
comments_df = comments_df.sample(n=1000, random_state=42).reset_index(drop=True)

# Drop explanation columns that are now completely empty
comments_df = comments_df.dropna(axis=1, how='all')

comments_df.to_csv('eli5_comments.csv', index=False)